# INDIA STARTUP MARKET OPPORTUNITY ENGINE
# Master Dataset Creation

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
 
# 1. FILE PATHS

DATA_DIR = Path("analysis and ML/data/raw")

STARTUP_FILE = DATA_DIR / "startups_dpiit_2016_2025.csv"
NSVA_FILE = DATA_DIR / "nsva_data.csv"
INTERNET_FILE = DATA_DIR / "internet_penetration.csv"
POPULATION_FILE = DATA_DIR / "population.csv"

OUTPUT_FILE = DATA_DIR / "india_startup_master_dataset.csv"

In [2]:
# 2. HELPER FUNCTIONS

def clean_column_names(df):
    """
    Standardize column names:
    - lowercase
    - remove leading/trailing whitespace
    - replace spaces/special characters with _
    """
    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    return df


def clean_state_name(x):
    """
    Standardize Indian state / UT names.

    Input can be uppercase, lowercase, contain extra spaces,
    punctuation, etc.
    """

    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper()

    # Normalize whitespace
    x = re.sub(r"\s+", " ", x)

    # Remove punctuation except &
    x = re.sub(r"[^\w\s&]", "", x)

    state_mapping = {

        "ANDHRA PRADESH": "Andhra Pradesh",
        "ARUNACHAL PRADESH": "Arunachal Pradesh",
        "ASSAM": "Assam",
        "BIHAR": "Bihar",
        "CHHATTISGARH": "Chhattisgarh",
        "GOA": "Goa",
        "GUJARAT": "Gujarat",
        "HARYANA": "Haryana",
        "HIMACHAL PRADESH": "Himachal Pradesh",
        "JHARKHAND": "Jharkhand",
        "KARNATAKA": "Karnataka",
        "KERALA": "Kerala",
        "MADHYA PRADESH": "Madhya Pradesh",
        "MAHARASHTRA": "Maharashtra",
        "MANIPUR": "Manipur",
        "MEGHALAYA": "Meghalaya",
        "MIZORAM": "Mizoram",
        "NAGALAND": "Nagaland",
        "ODISHA": "Odisha",
        "ORISSA": "Odisha",
        "PUNJAB": "Punjab",
        "RAJASTHAN": "Rajasthan",
        "SIKKIM": "Sikkim",
        "TAMIL NADU": "Tamil Nadu",
        "TELANGANA": "Telangana",
        "TRIPURA": "Tripura",
        "UTTAR PRADESH": "Uttar Pradesh",
        "UTTARAKHAND": "Uttarakhand",
        "WEST BENGAL": "West Bengal",

        # Union Territories
        "DELHI": "Delhi",
        "NCT OF DELHI": "Delhi",
        "JAMMU AND KASHMIR": "Jammu & Kashmir",
        "JAMMU KASHMIR": "Jammu & Kashmir",
        "LADAKH": "Ladakh",
        "CHANDIGARH": "Chandigarh",
        "PUDUCHERRY": "Puducherry",
        "PONDICHERRY": "Puducherry",
    }

    return state_mapping.get(x, x.title())


def clean_numeric(x):
    """
    Convert values such as:
        1,234
        Rs. 1,234
        ₹1,234
        1234.5
        -
        blank

    into numeric values.
    """

    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x in {"", "-", "--", "NA", "N/A", "NAN", "NULL"}:
        return np.nan

    # Remove commas and currency symbols/text
    x = x.replace(",", "")
    x = x.replace("₹", "")
    x = x.replace("Rs.", "")
    x = x.replace("Rs", "")

    # Extract first numeric value
    match = re.search(r"-?\d+(?:\.\d+)?", x)

    if match:
        return float(match.group())

    return np.nan


def extract_year(x):
    """
    Extract first 4-digit year from values such as:
        2024
        2024-25
        FY2024-25
        2024-2025
    """

    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    match = re.search(r"(19|20)\d{2}", x)

    if match:
        return int(match.group())

    return np.nan

In [3]:
# 3. LOAD STARTUP DATA

print("\nLoading startup data...")

startup = pd.read_csv(
    STARTUP_FILE,
    low_memory=False
)

startup = clean_column_names(startup)

print("Startup columns:")
print(startup.columns.tolist())


Loading startup data...
Startup columns:
['year', 'state', 'industry', 'startups_recognized', 'unit', 'note']


In [4]:
# 3A. IDENTIFY STARTUP COUNT COLUMN

startup = startup.rename(
    columns={
        "startups_rec": "startup_count",
        "startups_recognized": "startup_count",
        "startups_recognised": "startup_count",
        "startup_recognized": "startup_count",
        "startup_recognised": "startup_count"
    }
)


required_startup_columns = [
    "year",
    "state",
    "industry",
    "startup_count"
]

missing = [
    c for c in required_startup_columns
    if c not in startup.columns
]

if missing:
    raise ValueError(
        f"Missing startup columns: {missing}\n"
        f"Available columns: {startup.columns.tolist()}"
    )


startup = startup[
    required_startup_columns
].copy()

In [5]:
# 3B. CLEAN STARTUP DATA
 

startup["state"] = (
    startup["state"]
    .apply(clean_state_name)
)

startup["year"] = (
    startup["year"]
    .apply(extract_year)
)

startup["industry"] = (
    startup["industry"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

startup["startup_count"] = (
    startup["startup_count"]
    .apply(clean_numeric)
)


startup = startup.dropna(
    subset=[
        "year",
        "state",
        "industry",
        "startup_count"
    ]
)


startup["year"] = startup["year"].astype(int)

In [6]:
# 3C. AGGREGATE DUPLICATES
 

startup = (
    startup
    .groupby(
        ["year", "state", "industry"],
        as_index=False
    )["startup_count"]
    .sum()
)


print("\nClean startup data:")
print(startup.head())
print("Shape:", startup.shape)


Clean startup data:
   year           state                   industry  startup_count
0  2016  Andhra Pradesh                     Others            4.0
1  2016           Assam  Healthcare & Lifesciences            1.0
2  2016           Assam                     Others            9.0
3  2016           Bihar                     Others            1.0
4  2016      Chandigarh  Healthcare & Lifesciences            2.0
Shape: (9810, 4)


In [7]:
# 4. LOAD NSVA DATA

print("\nLoading NSVA data...")

nsva = pd.read_csv(
    NSVA_FILE,
    low_memory=False
)

nsva = clean_column_names(nsva)

print("NSVA columns:")
print(nsva.columns.tolist())


Loading NSVA data...
NSVA columns:
['year', 'state', 'nsva']


In [8]:
# 4A. VALIDATE NSVA COLUMNS


required_nsva_columns = [
    "year",
    "state",
    "nsva"
]

missing = [
    c for c in required_nsva_columns
    if c not in nsva.columns
]

if missing:
    raise ValueError(
        f"Missing NSVA columns: {missing}\n"
        f"Available columns: {nsva.columns.tolist()}"
    )


nsva = nsva[
    required_nsva_columns
].copy()

In [9]:
# 4B. CLEAN NSVA

nsva["state"] = (
    nsva["state"]
    .apply(clean_state_name)
)

nsva["year"] = (
    nsva["year"]
    .apply(extract_year)
)

nsva["nsva"] = (
    nsva["nsva"]
    .apply(clean_numeric)
)


nsva = nsva.dropna(
    subset=[
        "year",
        "state",
        "nsva"
    ]
)


nsva["year"] = nsva["year"].astype(int)

In [10]:
# 4C. AGGREGATE DUPLICATES

nsva = (
    nsva
    .groupby(
        ["year", "state"],
        as_index=False
    )["nsva"]
    .sum()
)


print("\nClean NSVA data:")
print(nsva.head())
print("Shape:", nsva.shape)

 
# 5. LOAD INTERNET PENETRATION DATA

print("\nLoading internet penetration data...")

internet = pd.read_csv(
    INTERNET_FILE,
    low_memory=False
)

internet = clean_column_names(internet)

print("Internet columns:")
print(internet.columns.tolist())


Clean NSVA data:
   year                        state      nsva
0  2011  Andaman And Nicobar Islands    3463.0
1  2011               Andhra Pradesh  310347.0
2  2011            Arunachal Pradesh   10021.0
3  2011                        Assam  121469.0
4  2011                        Bihar  223052.0
Shape: (476, 3)

Loading internet penetration data...
Internet columns:
['year', 'internet_penetration']


In [11]:
# 5A. VALIDATE INTERNET COLUMNS

required_internet_columns = [
    "year",
    "internet_penetration"
]

missing = [
    c for c in required_internet_columns
    if c not in internet.columns
]

if missing:
    raise ValueError(
        f"Missing internet columns: {missing}\n"
        f"Available columns: {internet.columns.tolist()}"
    )


internet = internet[
    required_internet_columns
].copy()

In [12]:
# 5B. CLEAN INTERNET DATA

internet["year"] = (
    internet["year"]
    .apply(extract_year)
)

internet["internet_penetration"] = (
    internet["internet_penetration"]
    .apply(clean_numeric)
)


internet = internet.dropna(
    subset=[
        "year",
        "internet_penetration"
    ]
)


internet["year"] = internet["year"].astype(int)

In [13]:
internet = internet.rename(
    columns={
        "internet_penetration":
        "india_internet_penetration"
    }
)


# If duplicate years exist, average them.
internet = (
    internet
    .groupby("year", as_index=False)
    ["india_internet_penetration"]
    .mean()
)


print("\nClean internet data:")
print(internet.head())
print("Shape:", internet.shape)


Clean internet data:
   year  india_internet_penetration
0  1992                    0.000111
1  1993                    0.000218
2  1994                    0.001070
3  1995                    0.026200
4  1996                    0.046300
Shape: (34, 2)


In [14]:
# 6. LOAD POPULATION DATA

print("\nLoading population data...")

population = pd.read_csv(
    POPULATION_FILE,
    low_memory=False
)

population = clean_column_names(population)

print("Population columns:")
print(population.columns.tolist())


Loading population data...
Population columns:
['year', 'population']


In [15]:
# 6A. VALIDATE POPULATION COLUMNS

required_population_columns = [
    "year",
    "population"
]

missing = [
    c for c in required_population_columns
    if c not in population.columns
]

if missing:
    raise ValueError(
        f"Missing population columns: {missing}\n"
        f"Available columns: {population.columns.tolist()}"
    )


population = population[
    required_population_columns
].copy()

In [16]:
# 6B. CLEAN POPULATION
 

population["year"] = (
    population["year"]
    .apply(extract_year)
)

population["population"] = (
    population["population"]
    .apply(clean_numeric)
)


population = population.dropna(
    subset=[
        "year",
        "population"
    ]
)


population["year"] = population["year"].astype(int)

In [17]:
population = population.rename(
    columns={
        "population":
        "india_population"
    }
)


# If duplicate years exist, keep the first.
population = (
    population
    .groupby("year", as_index=False)
    ["india_population"]
    .first()
)


print("\nClean population data:")
print(population.head())
print("Shape:", population.shape)


Clean population data:
   year  india_population
0  1960       435990338.0
1  1961       446564729.0
2  1962       457283090.0
3  1963       468138575.0
4  1964       479229598.0
Shape: (66, 2)


In [18]:
# 7. MERGE STARTUP + NSVA
 

print("\nMerging startup and NSVA data...")

master = startup.merge(
    nsva,
    on=[
        "year",
        "state"
    ],
    how="left",
    validate="many_to_one"
)


# 8. MERGE NATIONAL INTERNET DATA


master = master.merge(
    internet[
        [
            "year",
            "india_internet_penetration"
        ]
    ],
    on="year",
    how="left",
    validate="many_to_one"
)


# 9. MERGE NATIONAL POPULATION


master = master.merge(
    population[
        [
            "year",
            "india_population"
        ]
    ],
    on="year",
    how="left",
    validate="many_to_one"
)


Merging startup and NSVA data...


In [19]:
master = (
    master
    .sort_values(
        [
            "year",
            "state",
            "industry"
        ]
    )
    .reset_index(drop=True)
)


# 11. STATE-YEAR STARTUP TOTALS
 

master["state_total_startups"] = (
    master
    .groupby(
        [
            "year",
            "state"
        ]
    )["startup_count"]
    .transform("sum")
)


# 12. SECTOR SHARE


master["sector_share"] = np.where(
    master["state_total_startups"] > 0,

    master["startup_count"]
    / master["state_total_startups"],

    np.nan
)


master["sector_share_pct"] = (
    master["sector_share"] * 100
)


# 13. STATE STARTUP GROWTH
 

state_year = (
    master[
        [
            "year",
            "state",
            "state_total_startups"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "state",
            "year"
        ]
    )
)


state_year["startup_growth_pct"] = (
    state_year
    .groupby("state")
    ["state_total_startups"]
    .pct_change()
    * 100
)


master = master.merge(
    state_year[
        [
            "year",
            "state",
            "startup_growth_pct"
        ]
    ],
    on=[
        "year",
        "state"
    ],
    how="left",
    validate="many_to_one"
)

In [20]:
# 14. NATIONAL MACRO TRENDS
 

# IMPORTANT:
# Calculate internet growth ONCE per year first.
# Do not use diff() directly on the master table because
# each year appears multiple times across states/industries.

macro = (
    master[
        [
            "year",
            "india_internet_penetration"
        ]
    ]
    .drop_duplicates()
    .sort_values("year")
)


macro["india_internet_growth_pp"] = (
    macro["india_internet_penetration"]
    .diff()
)


master = master.merge(
    macro[
        [
            "year",
            "india_internet_growth_pp"
        ]
    ],
    on="year",
    how="left",
    validate="many_to_one"
)


# 15. OPTIONAL: NATIONAL STARTUP TOTAL
 

national_startups = (
    master
    .groupby("year")["startup_count"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "startup_count":
            "india_total_startups"
        }
    )
)


master = master.merge(
    national_startups,
    on="year",
    how="left",
    validate="many_to_one"
)


# 16. OPTIONAL: INDUSTRY SHARE NATIONALLY
 

industry_year = (
    master
    .groupby(
        [
            "year",
            "industry"
        ],
        as_index=False
    )["startup_count"]
    .sum()
)


industry_year = industry_year.merge(
    national_startups,
    on="year",
    how="left"
)


industry_year["industry_share_national"] = np.where(
    industry_year["india_total_startups"] > 0,

    industry_year["startup_count"]
    / industry_year["india_total_startups"],

    np.nan
)


master = master.merge(
    industry_year[
        [
            "year",
            "industry",
            "industry_share_national"
        ]
    ],
    on=[
        "year",
        "industry"
    ],
    how="left",
    validate="many_to_one"
)

In [21]:
# 18. DATA QUALITY CHECKS
 

print("\n==============================")
print("DATA QUALITY CHECK")
print("==============================")


print("\nMaster shape:")
print(master.shape)


print("\nYears:")
print(
    master["year"].min(),
    "to",
    master["year"].max()
)


print("\nNumber of states:")
print(
    master["state"].nunique()
)


print("\nNumber of industries:")
print(
    master["industry"].nunique()
)


print("\nStates:")
print(
    sorted(
        master["state"]
        .dropna()
        .unique()
    )
)


print("\nMissing values:")
print(
    master
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(20)
)


print("\nStartup totals by year:")
print(
    master
    .groupby("year")
    ["startup_count"]
    .sum()
    .tail(10)
)


print("\nNSVA coverage:")
print(
    master["nsva"]
    .notna()
    .mean() * 100,
    "%"
)


print("\nInternet coverage:")
print(
    master["india_internet_penetration"]
    .notna()
    .mean() * 100,
    "%"
)


print("\nPopulation coverage:")
print(
    master["india_population"]
    .notna()
    .mean() * 100,
    "%"
)


DATA QUALITY CHECK

Master shape:
(9810, 14)

Years:
2016 to 2025

Number of states:
36

Number of industries:
56

States:
['Andaman And Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra And Nagar Haveli And Daman And Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu & Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']

Missing values:
nsva                          967
startup_growth_pct             95
india_internet_growth_pp       64
year                            0
state                           0
industry                        0
startup_count                   0
india_internet_penetration      0
india_population                0
state_total_startups          

In [22]:
# 19. CHECK FOR DUPLICATE MASTER KEYS
 

duplicate_keys = (
    master
    .duplicated(
        subset=[
            "year",
            "state",
            "industry"
        ],
        keep=False
    )
)


print("\nDuplicate year-state-industry rows:")

print(
    duplicate_keys.sum()
)


if duplicate_keys.any():

    print(
        master.loc[
            duplicate_keys,
            [
                "year",
                "state",
                "industry"
            ]
        ]
        .head(20)
    )

master = master.dropna()


Duplicate year-state-industry rows:
0


In [23]:
# 20. SAVE MASTER DATASET
 

master.to_csv(
    OUTPUT_FILE,
    index=False
)


print("\n================================")
print("MASTER DATASET CREATED")
print("================================")

print(
    f"Saved to: {OUTPUT_FILE}"
)

# 21. FINAL COLUMNS
 

print("\nFinal columns:")

for col in master.columns:
    print(" -", col)

# 22. PREVIEW
 

print("\nPreview:")

print(
    master
    .head(20)
    .to_string(index=False)
)


MASTER DATASET CREATED
Saved to: analysis and ML/data/raw/india_startup_master_dataset.csv

Final columns:
 - year
 - state
 - industry
 - startup_count
 - nsva
 - india_internet_penetration
 - india_population
 - state_total_startups
 - sector_share
 - sector_share_pct
 - startup_growth_pct
 - india_internet_growth_pp
 - india_total_startups
 - industry_share_national

Preview:
 year          state                            industry  startup_count     nsva  india_internet_penetration  india_population  state_total_startups  sector_share  sector_share_pct  startup_growth_pct  india_internet_growth_pp  india_total_startups  industry_share_national
 2017 Andhra Pradesh                                  AI            2.0 645027.0                   18.200001      1359657400.0                 103.0      0.019417          1.941748              2475.0                  1.700001                5473.0                 0.014069
 2017 Andhra Pradesh AR VR (Augmented + Virtual Reality)            1